In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/avito_retrieval"):
        !git clone https://github.com/mishin-mikhail/avito_retrieval /content/avito_retrieval
    else:
        !cd /content/avito_retrieval && git pull
    %cd /content/avito_retrieval/notebooks
    !pip -q install sentence-transformers rank-bm25 snowballstemmer scikit-learn pyarrow beautifulsoup4 lxml
    import torch
    print("CUDA:", torch.cuda.is_available())

Cloning into '/content/avito-retrieval'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 52 (delta 23), reused 18 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 2.92 MiB | 5.37 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/avito-retrieval/notebooks
Mounted at /content/drive
CUDA: True


## Конфигурация



In [3]:
#Препроцессинг
TITLE_BOOST = 3
CHUNK_WORDS = 160
STRIDE_WORDS = 120

#Модели
ENCODER_NAME = "intfloat/multilingual-e5-large"
ENCODER2_NAME = "BAAI/bge-m3"
RERANKER_NAME = "BAAI/bge-reranker-v2-m3"

#Слияние (weighted RRF)
RRF_K = 40
WEIGHTS = [1.0, 2.5, 0.5, 5.0, 2.0]
CANDIDATES = 30

#Реранкер
BLEND_DEPTH = 15
W_CE = 0.5

#Прочее
SEED = 21
N_FOLDS = 5
RUN_SANITY = True

## Данные и препроцессинг

In [4]:
sys.path.append("..")
from pathlib import Path

import numpy as np
import pandas as pd

from src.common import (
    clean_html, tokenize, chunk_text,
    map_at_k, parse_ground_truth,
)

DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

articles = pd.read_feather(DATA / "articles.f")
calib = pd.read_feather(DATA / "calibration.f")
test = pd.read_feather(DATA / "test.f")

articles["clean_body"] = articles["body"].map(clean_html)
article_ids = articles["article_id"].to_numpy()
id2pos = {a: i for i, a in enumerate(article_ids)}
calib_queries = calib["query_text"].tolist()
test_queries = test["query_text"].tolist()
truth = {r.query_id: parse_ground_truth(r.ground_truth) for r in calib.itertuples()}
print(f"{len(articles)} статей, {len(calib)} calibration, {len(test)} test")

793 статей, 500 calibration, 500 test


## Каналы ранжирования

In [5]:
%%time
#Канал A: BM25
from rank_bm25 import BM25Okapi

doc_text = ((articles["title"].fillna("") + " ") * TITLE_BOOST + articles["clean_body"]).tolist()
bm25 = BM25Okapi([tokenize(t) for t in doc_text])

def bm25_rankings(queries):
    return np.stack([np.argsort(-np.round(bm25.get_scores(tokenize(q)), 5), kind="stable")
                     for q in queries])

#Канал B: символьные n-граммы
from sklearn.feature_extraction.text import TfidfVectorizer

char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
char_doc_matrix = char_vec.fit_transform(doc_text)

def char_rankings(queries):
    sims = np.round((char_vec.transform(queries) @ char_doc_matrix.T).toarray(), 5)
    return np.argsort(-sims, axis=1, kind="stable")

CPU times: user 22.4 s, sys: 227 ms, total: 22.7 s
Wall time: 22.8 s


In [6]:
%%time
#Каналы C и E: два bi-encoder'а на общих чанках
from sentence_transformers import SentenceTransformer

chunk_texts, chunk_owner = [], []
for pos, row in enumerate(articles.itertuples()):
    title = row.title or ""
    for ch in chunk_text(row.clean_body, chunk_words=CHUNK_WORDS, stride_words=STRIDE_WORDS):
        chunk_texts.append(f"{title}. {ch}")
        chunk_owner.append(pos)
chunk_owner = np.array(chunk_owner)

def art_rank_from_chunk_sims(sims):
    sims = np.round(sims, 5)
    n_art = len(article_ids)
    art_scores = np.empty((sims.shape[0], n_art))
    best_chunk = np.empty((sims.shape[0], n_art), dtype=int)
    for a in range(n_art):
        idx = np.flatnonzero(chunk_owner == a)
        block = sims[:, idx]
        art_scores[:, a] = block.max(axis=1)
        best_chunk[:, a] = idx[block.argmax(axis=1)]
    return np.argsort(-art_scores, axis=1, kind="stable"), best_chunk

def load_or_encode(model, name, passages, calib_texts, test_texts):
    tag = name.split("/")[-1]
    p_chunks, p_queries = CACHE / f"chunks_{tag}.npy", CACHE / f"queries_{tag}.npz"
    if p_chunks.exists():
        c_emb = np.load(p_chunks)
        assert len(c_emb) == len(chunk_texts)
    else:
        c_emb = model.encode(passages, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
        np.save(p_chunks, c_emb)
    if p_queries.exists():
        z = np.load(p_queries)
        cq, tq = z["calib"], z["test"]
    else:
        cq = model.encode(calib_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
        tq = model.encode(test_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
        np.savez(p_queries, calib=cq, test=tq)
    return c_emb, cq, tq

#C: e5-large - с префиксами query:/passage:
encoder = SentenceTransformer(ENCODER_NAME)
print("e5 device:", encoder.device)
e5_chunk_emb, e5_calib_q, e5_test_q = load_or_encode(
    encoder, ENCODER_NAME,
    [f"passage: {t}" for t in chunk_texts],
    [f"query: {t}" for t in calib_queries],
    [f"query: {t}" for t in test_queries],
)

#E: bge-m3 - без префиксов
encoder2 = SentenceTransformer(ENCODER2_NAME)
bge_chunk_emb, bge_calib_q, bge_test_q = load_or_encode(
    encoder2, ENCODER2_NAME, chunk_texts, calib_queries, test_queries,
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

CPU times: user 43.6 s, sys: 19 s, total: 1min 2s
Wall time: 1min 54s


In [7]:
#Канал D: kNN по размеченным запросам calibration (эмбеддинги e5)
gt_member = np.zeros((len(calib), len(article_ids)), dtype=bool)
for i, r in enumerate(calib.itertuples()):
    for a in parse_ground_truth(r.ground_truth):
        gt_member[i, id2pos[a]] = True

folds = rng.permutation(len(calib)) % N_FOLDS

def knn_scores(q_emb, mask_fold=None):
    sims = q_emb @ e5_calib_q.T
    if mask_fold is not None:
        sims = np.where(mask_fold[:, None] == folds[None, :], -np.inf, sims)
    scores = np.full((len(q_emb), len(article_ids)), -np.inf)
    for a in range(len(article_ids)):
        idx = np.flatnonzero(gt_member[:, a])
        if len(idx):
            scores[:, a] = sims[:, idx].max(axis=1)
    return np.round(scores, 5)

## Слияние, реранкер, бленд

In [8]:
%%time
from sentence_transformers import CrossEncoder

def rrf_fuse(rank_mats, weights, k=RRF_K, top_n=CANDIDATES):
    n_q, n_art = rank_mats[0].shape
    rows = np.arange(n_q)[:, None]
    fused = np.zeros((n_q, n_art))
    for R, w in zip(rank_mats, weights):
        pos = np.empty_like(R)
        pos[rows, R] = np.arange(n_art)
        fused = fused + w / (k + 1 + pos)
    return np.argsort(-fused, axis=1, kind="stable")[:, :top_n]

reranker = CrossEncoder(RERANKER_NAME, max_length=512)
print("reranker device:", reranker.model.device)

def ce_scores(queries, cand_idx, best_chunk, cache_name):
    cache_file = CACHE / f"rerank_{cache_name}.npz"
    if cache_file.exists():
        z = np.load(cache_file)
        if z["cand"].shape == cand_idx.shape and (z["cand"] == cand_idx).all():
            return z["scores"]
    pairs = [(queries[q], chunk_texts[best_chunk[q, a]])
             for q in range(len(queries)) for a in cand_idx[q]]
    scores = np.asarray(reranker.predict(pairs, batch_size=32, show_progress_bar=True))
    scores = scores.reshape(len(queries), cand_idx.shape[1])
    np.savez(cache_file, cand=cand_idx, scores=scores)
    return scores

def blend_rank(cand_idx, ce, w_ce=W_CE, depth=BLEND_DEPTH, k=RRF_K):
    out = []
    for q in range(len(cand_idx)):
        head = cand_idx[q, :depth]
        ce_pos = np.empty(depth, dtype=int)
        ce_pos[np.argsort(-np.round(ce[q, :depth], 5), kind="stable")] = np.arange(depth)
        score = 1.0 / (k + 1 + np.arange(depth)) + w_ce / (k + 1 + ce_pos)
        order = head[np.argsort(-score, kind="stable")]
        tail = [i for i in cand_idx[q] if i not in set(order)]
        out.append([int(article_ids[i]) for i in list(order) + tail][:10])
    return out

def run_pipeline(queries, e5_q, bge_q, knn_mask, cache_name):
    rank_e5, best_chunk = art_rank_from_chunk_sims(e5_q @ e5_chunk_emb.T)
    mats = [
        bm25_rankings(queries),
        rank_e5,
        char_rankings(queries),
        np.argsort(-knn_scores(e5_q, mask_fold=knn_mask), axis=1, kind="stable"),
        art_rank_from_chunk_sims(bge_q @ bge_chunk_emb.T)[0],
    ]
    cand_idx = rrf_fuse(mats, WEIGHTS)
    ce = ce_scores(queries, cand_idx, best_chunk, cache_name)
    return blend_rank(cand_idx, ce)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

reranker device: cuda:0
CPU times: user 9.27 s, sys: 5.21 s, total: 14.5 s
Wall time: 47 s


## Воспроизводим метрику на calibration

kNN-канал с 5-fold маскированием.

In [9]:
%%time
if RUN_SANITY:
    calib_pred = run_pipeline(calib_queries, e5_calib_q, bge_calib_q,
                              knn_mask=folds, cache_name="calib_final5")
    score = map_at_k(dict(zip(calib["query_id"], calib_pred)), truth, k=10)
    print(f"MAP@10 на calibration (5-fold для kNN): {score:.4f}")

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

MAP@10 на calibration (5-fold для kNN): 0.6448
CPU times: user 13min 53s, sys: 1.24 s, total: 13min 54s
Wall time: 14min 3s


## Анализ ошибок на calibration

Худшие запросы по AP@10.

In [10]:
if RUN_SANITY:
    from src.common import ap_at_k
    title_by_id = dict(zip(articles["article_id"], articles["title"]))
    preds = dict(zip(calib["query_id"], calib_pred))
    rows = []
    for r in calib.itertuples():
        rel = truth[r.query_id]
        rows.append({
            "query_id": r.query_id,
            "ap@10": ap_at_k(preds[r.query_id], rel, k=10),
            "query": r.query_text,
            "truth": [f"{a}: {title_by_id.get(a, '?')}" for a in rel],
            "top5_pred": [f"{a}: {title_by_id.get(a, '?')}" for a in preds[r.query_id][:5]],
        })
    errors = pd.DataFrame(rows).sort_values("ap@10")
    print("AP=0 (полный промах):", (errors["ap@10"] == 0).mean().round(3))
    print("AP=1 (идеальный ответ):", (errors["ap@10"] == 1).mean().round(3))
    pd.set_option("display.max_colwidth", 120)
    display(errors.head(10))

AP=0 (полный промах): 0.036
AP=1 (идеальный ответ): 0.38


,query_id,ap@10,query,truth,top5_pred
478,479,0.0,"Подскажите, я хочу выставить объявление по услугам риелтора на бали, в какую категорию мне лучше выставлять его?",[4224: Предложить услугу],"[3993: Нужно перенести объявление в Недвижимость, 4276: Тариф с оплатой размещений для Недвижимости, 4321: Мой урове..."
80,81,0.0,Добрый день. Разьясните пжл по брони с 10 по <DATE>. Что это за акция? Сколько мне гость при заселении должен? Там и...,[3478: Акция от Авито: скидка на комиссию за бронь],"[4506: Частые вопросы, 4508: Частые вопросы, 3314: Настроить скидки за бронь, 4505: Добавить инструкцию по заселению..."
454,455,0.0,Можно ли заказать Авито доставку от Сестрорецка до Пушкина?,[1899: В каких городах есть доставка],"[4308: Заказать товар с доставкой, 4286: Грузовая доставка, 4234: Как продавать и покупать с доставкой, 4328: Что мо..."
452,453,0.0,Эффективный Черчилль: Дмитрий Медведев Здравствуйте! Эта книга разрешена к продаже на площадке?,[4133: Ограничены звонки и чаты с покупателями],"[4328: Что можно заказать и отправить, 4307: Лимит бесплатных размещений, 4320: Как поддерживать достоверность цен в..."
324,325,0.0,"Здравствуйте, что делать, если некорректно выполнили работу, которую предоставили через Авито, взяли деньги, но пере...","[4332: Меня обманули, 4285: Как работает Защита сделки: информация для заказчиков]","[4440: Платёж прошёл, но есть проблема, 4219: Покупателю, 4400: Покупателю — отказаться от товара или вернуть его, 2..."
255,256,0.0,"Добрый день. Подскажите пожалуйста, а где мои 10тысяч которые я вывел на т-банк себе?",[2943: Не могу вывести деньги за доставку],"[4384: Баланс для покупок, 4440: Платёж прошёл, но есть проблема, 4219: Покупателю, 4332: Меня обманули, 1966: Когда..."
253,254,0.0,"Заказ <ID> Скажите, когда мне вернутся деньги?",[4218: Тариф с оплатой за просмотры в Товарах],"[2865: Когда вернутся деньги за доставку, 4219: Покупателю, 1966: Когда вернутся деньги после отмены доставки в пунк..."
98,99,0.0,"здравствуйте. мне дает вновь опубликовать истекшие, система пишет заполнить обязательные поля, но все поля заполнены...",[2202: Не сохраняются изменения],"[4307: Лимит бесплатных размещений, 4440: Платёж прошёл, но есть проблема, 2886: Отклонили оплаченное объявление с в..."
387,388,0.0,"здравствуйте, можно получить скидку на выставление второй квартиры в продажу?",[4276: Тариф с оплатой размещений для Недвижимости],"[4424: Бонусы Авито, 2698: Как продавцу подключить скидки, 4451: Просто скидка, 4395: Бонусы, 4307: Лимит бесплатных..."
307,308,0.0,Куда опять пропал Портал призов?,"[4321: Мой уровень сервиса, 4127: Уровень сервиса в Путешествиях]","[4219: Покупателю, 4389: Платёж не прошёл, 4214: Скидки, бонусы и промокоды, 3843: Заказ потерялся: что делать?, 266..."


## Инференс на test и answer.csv

Для test kNN-канал использует всю calibration. Test-запросы - только вход инференса.

In [11]:
%%time
test_pred = run_pipeline(test_queries, e5_test_q, bge_test_q,
                         knn_mask=None, cache_name="test_final5")

answer = test[["query_id"]].copy()
answer["answer"] = [" ".join(map(str, ids)) for ids in test_pred]
answer.to_csv("../answer.csv", index=False)
answer.head()

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

CPU times: user 13min 57s, sys: 923 ms, total: 13min 57s
Wall time: 13min 57s


,query_id,answer
0,1,4308 4396 2665 4440 3565 4219 4433 2222 4234 4283
1,2,4234 2646 4219 4361 4308 4400 4009 4403 4286 1966
2,3,1909 4234 4396 4308 4400 4219 4328 4286 2646 4403
3,4,4400 4532 2865 4219 4234 2646 4361 1966 4403 4308
4,5,3467 4234 4214 2698 4308 4361 1951 4320 4219 4328


In [13]:
#Валидация формата
sub = pd.read_csv("../answer.csv")
known_ids = set(articles["article_id"])
assert list(sub.columns) == ["query_id", "answer"], "колонки: query_id, answer"
assert len(sub) == len(test), "число строк = числу test-запросов"
assert set(sub["query_id"]) == set(test["query_id"]), "все query_id из test, без лишних"
assert sub["query_id"].is_unique, "без повторов query_id"
for _, r in sub.iterrows():
    ids = [int(x) for x in str(r["answer"]).split()]
    assert 1 <= len(ids) <= 10, "не больше 10 статей"
    assert len(ids) == len(set(ids)), "без повторов article_id внутри ответа"
    assert all(i in known_ids for i in ids), "все article_id существуют в articles.f"
print("answer.csv валиден по всем требованиям")

answer.csv валиден по всем требованиям
